# Zadanie 1 — Toy Example mapa 3

Ten sam świat co w `Toy_Example_mapa2_filtru_1D.ipynb`, inna sekwencja pomiarów.

```text
świat = [ ściana, ściana, drzwi, ściana, drzwi ]
```

Pomiary: **drzwi → ściana → drzwi → ściana** (prawdziwy start na polu **2**).

Trasa robota:

```text
2 (drzwi) → 3 (ściana) → 4 (drzwi) → 0 (ściana)
```


## Ustawienia świata

```text
świat = [ ściana, ściana, drzwi, ściana, drzwi ]
```

- Robot: ruch w prawo `+1 (mod 5)`
- Sensor: rozróżnia drzwi i ścianę z dokładnością `0.8`
- Sekwencja pomiarów: drzwi → ściana → drzwi → ściana
- Start prawdziwego robota: pole **2** (drzwi)


In [26]:
import numpy as np
import matplotlib.pyplot as plt

sciana = 0
drzwi = 1

swiat = np.array([sciana, sciana, drzwi, sciana, drzwi])
nazwy = {sciana: "ściana", drzwi: "drzwi"}

p_poprawny = 0.8
p_bledny = 0.2

print("świat =", [nazwy[x] for x in swiat])
print("indeksy:", list(range(len(swiat))))
print("drzwi na polach:", [i for i, x in enumerate(swiat) if x == drzwi])
print("ściany na polach:", [i for i, x in enumerate(swiat) if x == sciana])


świat = ['ściana', 'ściana', 'drzwi', 'ściana', 'drzwi']
indeksy: [0, 1, 2, 3, 4]
drzwi na polach: [2, 4]
ściany na polach: [0, 1, 3]


---
## Krok 1 — Inicjalizacja

Generujemy 10 cząsteczek w sposób równomierny:
`[0, 1, 2, 3, 4, 0, 1, 2, 3, 4]`


In [27]:
czasteczki = np.array([0, 1, 2, 3, 4, 0, 1, 2, 3, 4])
print("cząsteczki =", czasteczki.tolist())
print("typy pól  =", [nazwy[swiat[i]] for i in czasteczki])


cząsteczki = [0, 1, 2, 3, 4, 0, 1, 2, 3, 4]
typy pól  = ['ściana', 'ściana', 'drzwi', 'ściana', 'drzwi', 'ściana', 'ściana', 'drzwi', 'ściana', 'drzwi']


---
## Krok 2 — Pierwszy pomiar: drzwi

Wagi:
- `0.8` jeśli pole to drzwi
- `0.2` w przeciwnym razie


In [28]:
czasteczki = np.array([0, 1, 2, 3, 4, 0, 1, 2, 3, 4])  # Krok 1

pomiar = drzwi

w = []
for pozycja in czasteczki:
    if swiat[pozycja] == pomiar:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)

S = w.sum()
w_prim = np.round(w / S, 3)

print("2.a")
print("S =", round(S, 2))
print("\n")

print("2.b")
print("w  (przed normalizacją) =", np.round(w, 3).tolist())

print("w' (po normalizacji)    =", w_prim.tolist())
print(f"suma w'= {round(w_prim.sum(), 3)} ≈ 1")


2.a
S = 4.4


2.b
w  (przed normalizacją) = [0.2, 0.2, 0.8, 0.2, 0.8, 0.2, 0.2, 0.8, 0.2, 0.8]
w' (po normalizacji)    = [0.045, 0.045, 0.182, 0.045, 0.182, 0.045, 0.045, 0.182, 0.045, 0.182]
suma w'= 0.998 ≈ 1


### Resampling (ruletka, u i CDF)

Losowe wartości:
`u = [0.05, 0.18, 0.32, 0.41, 0.55, 0.62, 0.71, 0.86, 0.93, 0.99]`

Dla każdego `u_j` wybieramy **najmniejszy indeks i**, dla którego `CDF_i ≥ u_j`.


In [29]:
def zbuduj_CDF(wagi):
    return np.cumsum(wagi)


def resampling_ruletka(czasteczki, wagi, u):
    """Metoda ruletkowa jak w PDF: CDF + wartości u."""
    CDF = zbuduj_CDF(wagi)
    indeksy = []
    for uj in u:
        i = int(np.searchsorted(CDF, uj, side="left"))
        i = min(i, len(czasteczki) - 1)
        indeksy.append(i)
    indeksy = np.array(indeksy)
    nowe_czasteczki = czasteczki[indeksy].copy()
    return nowe_czasteczki, indeksy, CDF


czasteczki = np.array([0, 1, 2, 3, 4, 0, 1, 2, 3, 4])
w = []
for pozycja in czasteczki:
    if swiat[pozycja] == drzwi:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)
w_prim = np.round(w / w.sum(), 3)

u = np.array([0.05, 0.18, 0.32, 0.41, 0.55, 0.62, 0.71, 0.86, 0.93, 0.99])
czasteczki, indeksy, CDF = resampling_ruletka(czasteczki, w_prim, u)

print("2.c Resampling (ruletka/CDF)")
print("w'  =", w_prim.tolist())
print("CDF ≈", np.round(CDF, 3).tolist())
print("wybrane indeksy i =", indeksy.tolist())
print("nowe cząsteczki   =", czasteczki.tolist())

from IPython.display import Markdown, display

linie = [
    "**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)",
    "",
    "| $u_j$ | Przedział CDF | Wybrany indeks $i$ |",
    "|------:|:--------------|-------------------:|",
]
for uj, i in zip(u, indeksy):
    prawy = float(CDF[i])
    if i == 0:
        przedzial = f"u ≤ {prawy:.3f}"
    else:
        lewy = float(CDF[i - 1])
        przedzial = f"{lewy:.3f} < u ≤ {prawy:.3f}"
    linie.append(f"| {uj} | {przedzial} | {int(i)} |")

display(Markdown("\n".join(linie)))


2.c Resampling (ruletka/CDF)
w'  = [0.045, 0.045, 0.182, 0.045, 0.182, 0.045, 0.045, 0.182, 0.045, 0.182]
CDF ≈ [0.045, 0.09, 0.272, 0.317, 0.499, 0.544, 0.589, 0.771, 0.816, 0.998]
wybrane indeksy i = [1, 2, 4, 4, 6, 7, 7, 9, 9, 9]
nowe cząsteczki   = [1, 2, 4, 4, 1, 2, 2, 4, 4, 4]


**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)

| $u_j$ | Przedział CDF | Wybrany indeks $i$ |
|------:|:--------------|-------------------:|
| 0.05 | 0.045 < u ≤ 0.090 | 1 |
| 0.18 | 0.090 < u ≤ 0.272 | 2 |
| 0.32 | 0.317 < u ≤ 0.499 | 4 |
| 0.41 | 0.317 < u ≤ 0.499 | 4 |
| 0.55 | 0.544 < u ≤ 0.589 | 6 |
| 0.62 | 0.589 < u ≤ 0.771 | 7 |
| 0.71 | 0.589 < u ≤ 0.771 | 7 |
| 0.86 | 0.816 < u ≤ 0.998 | 9 |
| 0.93 | 0.816 < u ≤ 0.998 | 9 |
| 0.99 | 0.816 < u ≤ 0.998 | 9 |

| | |
|--|--|
| wybrane i | `[1, 2, 4, 4, 6, 7, 7, 9, 9, 9]` |
| indeksy | `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]` |
| pozycja na świecie | `[0, 1, 2, 3, 4, 0, 1, 2, 3, 4]` |
| nowe cząsteczki | `[1, 2, 4, 4, 1, 2, 2, 4, 4, 4]` |


---
## Krok 3 — Pierwszy ruch: +1 w prawo

`cząsteczki = (cząsteczki + 1) mod 5`


In [30]:
def move(czasteczki, rozmiar_swiata, krok=1):
    return (czasteczki + krok) % rozmiar_swiata


def rozklad(czasteczki, rozmiar_swiata):
    counts = np.bincount(czasteczki, minlength=rozmiar_swiata)
    return {i: int(c) for i, c in enumerate(counts) if c > 0}


czasteczki = np.array([1, 2, 4, 4, 1, 2, 2, 4, 4, 4])  # po kroku 2.c

print("przed ruchem:", czasteczki.tolist())
czasteczki = move(czasteczki, len(swiat))
print("po ruchu +1: ", czasteczki.tolist())
print("rozkład:     ", rozklad(czasteczki, len(swiat)))


przed ruchem: [1, 2, 4, 4, 1, 2, 2, 4, 4, 4]
po ruchu +1:  [2, 3, 0, 0, 2, 3, 3, 0, 0, 0]
rozkład:      {0: 5, 2: 2, 3: 3}


---
## Krok 4 — Drugi pomiar: ściana

Ściana na pozycjach `{0, 1, 3}`.

Wagi:
- `0.8` jeśli pole to ściana
- `0.2` w przeciwnym razie


In [31]:
czasteczki = np.array([2, 3, 0, 0, 2, 3, 3, 0, 0, 0])  # po kroku 3

pomiar = sciana

w = []
for pozycja in czasteczki:
    if swiat[pozycja] == pomiar:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)

S = w.sum()
w_prim = np.round(w / S, 3)

print("4.a")
print("S =", round(S, 2))
print("\n")

print("4.b")
print("w  (przed normalizacją) =", np.round(w, 3).tolist())

print("w' (po normalizacji)    =", w_prim.tolist())
print(f"suma w'= {round(w_prim.sum(), 3)} ≈ 1")


4.a
S = 6.8


4.b
w  (przed normalizacją) = [0.2, 0.8, 0.8, 0.8, 0.2, 0.8, 0.8, 0.8, 0.8, 0.8]
w' (po normalizacji)    = [0.029, 0.118, 0.118, 0.118, 0.029, 0.118, 0.118, 0.118, 0.118, 0.118]
suma w'= 1.002 ≈ 1


### Resampling (ruletka, u i CDF)

Losowe wartości:
`u = [0.05, 0.12, 0.19, 0.24, 0.41, 0.43, 0.74, 0.88, 0.93, 0.97]`

Dla każdego `u_j` wybieramy **najmniejszy indeks i**, dla którego `CDF_i ≥ u_j`.


In [32]:
czasteczki = np.array([2, 3, 0, 0, 2, 3, 3, 0, 0, 0])
w = []
for pozycja in czasteczki:
    if swiat[pozycja] == sciana:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)
w_prim = np.round(w / w.sum(), 3)

u = np.array([0.05, 0.12, 0.19, 0.24, 0.41, 0.43, 0.74, 0.88, 0.93, 0.97])
czasteczki, indeksy, CDF = resampling_ruletka(czasteczki, w_prim, u)

print("4.c Resampling (ruletka/CDF)")
print("w'  =", w_prim.tolist())
print("CDF ≈", np.round(CDF, 3).tolist())
print("wybrane indeksy i =", indeksy.tolist())
print("nowe cząsteczki   =", czasteczki.tolist())

from IPython.display import Markdown, display

linie = [
    "**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)",
    "",
    "| $u_j$ | Przedział CDF | Wybrany indeks $i$ |",
    "|------:|:--------------|-------------------:|",
]
for uj, i in zip(u, indeksy):
    prawy = float(CDF[i])
    if i == 0:
        przedzial = f"u ≤ {prawy:.3f}"
    else:
        lewy = float(CDF[i - 1])
        przedzial = f"{lewy:.3f} < u ≤ {prawy:.3f}"
    linie.append(f"| {uj} | {przedzial} | {int(i)} |")

display(Markdown("\n".join(linie)))


4.c Resampling (ruletka/CDF)
w'  = [0.029, 0.118, 0.118, 0.118, 0.029, 0.118, 0.118, 0.118, 0.118, 0.118]
CDF ≈ [0.029, 0.147, 0.265, 0.383, 0.412, 0.53, 0.648, 0.766, 0.884, 1.002]
wybrane indeksy i = [1, 1, 2, 2, 4, 5, 7, 8, 9, 9]
nowe cząsteczki   = [3, 3, 0, 0, 2, 3, 0, 0, 0, 0]


**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)

| $u_j$ | Przedział CDF | Wybrany indeks $i$ |
|------:|:--------------|-------------------:|
| 0.05 | 0.029 < u ≤ 0.147 | 1 |
| 0.12 | 0.029 < u ≤ 0.147 | 1 |
| 0.19 | 0.147 < u ≤ 0.265 | 2 |
| 0.24 | 0.147 < u ≤ 0.265 | 2 |
| 0.41 | 0.383 < u ≤ 0.412 | 4 |
| 0.43 | 0.412 < u ≤ 0.530 | 5 |
| 0.74 | 0.648 < u ≤ 0.766 | 7 |
| 0.88 | 0.766 < u ≤ 0.884 | 8 |
| 0.93 | 0.884 < u ≤ 1.002 | 9 |
| 0.97 | 0.884 < u ≤ 1.002 | 9 |

**Wynik resamplingu (krok 4):**

| | |
|--|--|
| wybrane i | `[1, 1, 2, 2, 4, 5, 7, 8, 9, 9]` |
| indeksy | `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]` |
| pozycja na świecie | `[2, 3, 0, 0, 2, 3, 3, 0, 0, 0]` |
| nowe cząsteczki | `[3, 3, 0, 0, 2, 3, 0, 0, 0, 0]` |


---
## Krok 5 — Drugi ruch: +1 w prawo

`cząsteczki = (cząsteczki + 1) mod 5`


In [33]:
czasteczki = np.array([3, 3, 0, 0, 2, 3, 0, 0, 0, 0])  # po kroku 4

print("przed ruchem:", czasteczki.tolist())
czasteczki = move(czasteczki, len(swiat))
print("po ruchu +1: ", czasteczki.tolist())
print("rozkład:     ", rozklad(czasteczki, len(swiat)))


przed ruchem: [3, 3, 0, 0, 2, 3, 0, 0, 0, 0]
po ruchu +1:  [4, 4, 1, 1, 3, 4, 1, 1, 1, 1]
rozkład:      {1: 6, 3: 1, 4: 3}


---
## Krok 6 — Trzeci pomiar: drzwi

Drzwi na pozycjach `{2, 4}`.

Wagi:
- `0.8` jeśli pole to drzwi
- `0.2` w przeciwnym razie


In [34]:
czasteczki = np.array([4, 4, 1, 1, 3, 4, 1, 1, 1, 1])  # po kroku 5

pomiar = drzwi

w = []
for pozycja in czasteczki:
    if swiat[pozycja] == pomiar:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)

S = w.sum()
w_prim = np.round(w / S, 3)

print("6.a")
print("S =", round(S, 2))
print("\n")

print("6.b")
print("w  (przed normalizacją) =", np.round(w, 3).tolist())

print("w' (po normalizacji)    =", w_prim.tolist())
print(f"suma w'= {round(w_prim.sum(), 3)} ≈ 1")


6.a
S = 3.8


6.b
w  (przed normalizacją) = [0.8, 0.8, 0.2, 0.2, 0.2, 0.8, 0.2, 0.2, 0.2, 0.2]
w' (po normalizacji)    = [0.211, 0.211, 0.053, 0.053, 0.053, 0.211, 0.053, 0.053, 0.053, 0.053]
suma w'= 1.004 ≈ 1


### Resampling (ruletka, u i CDF)

Losowe wartości:
`u = [0.05, 0.12, 0.20, 0.33, 0.45, 0.60, 0.72, 0.81, 0.93, 0.99]`

Dla każdego `u_j` wybieramy **najmniejszy indeks i**, dla którego `CDF_i ≥ u_j`.


In [35]:
czasteczki = np.array([4, 4, 1, 1, 3, 4, 1, 1, 1, 1])
w = []
for pozycja in czasteczki:
    if swiat[pozycja] == drzwi:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)
w_prim = np.round(w / w.sum(), 3)

u = np.array([0.05, 0.12, 0.20, 0.33, 0.45, 0.60, 0.72, 0.81, 0.93, 0.99])
czasteczki, indeksy, CDF = resampling_ruletka(czasteczki, w_prim, u)

print("6.c Resampling (ruletka/CDF)")
print("w'  =", w_prim.tolist())
print("CDF ≈", np.round(CDF, 3).tolist())
print("wybrane indeksy i =", indeksy.tolist())
print("nowe cząsteczki   =", czasteczki.tolist())

from IPython.display import Markdown, display

linie = [
    "**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)",
    "",
    "| $u_j$ | Przedział CDF | Wybrany indeks $i$ |",
    "|------:|:--------------|-------------------:|",
]
for uj, i in zip(u, indeksy):
    prawy = float(CDF[i])
    if i == 0:
        przedzial = f"u ≤ {prawy:.3f}"
    else:
        lewy = float(CDF[i - 1])
        przedzial = f"{lewy:.3f} < u ≤ {prawy:.3f}"
    linie.append(f"| {uj} | {przedzial} | {int(i)} |")

display(Markdown("\n".join(linie)))


6.c Resampling (ruletka/CDF)
w'  = [0.211, 0.211, 0.053, 0.053, 0.053, 0.211, 0.053, 0.053, 0.053, 0.053]
CDF ≈ [0.211, 0.422, 0.475, 0.528, 0.581, 0.792, 0.845, 0.898, 0.951, 1.004]
wybrane indeksy i = [0, 0, 0, 1, 2, 5, 5, 6, 8, 9]
nowe cząsteczki   = [4, 4, 4, 4, 1, 4, 4, 1, 1, 1]


**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)

| $u_j$ | Przedział CDF | Wybrany indeks $i$ |
|------:|:--------------|-------------------:|
| 0.05 | u ≤ 0.211 | 0 |
| 0.12 | u ≤ 0.211 | 0 |
| 0.2 | u ≤ 0.211 | 0 |
| 0.33 | 0.211 < u ≤ 0.422 | 1 |
| 0.45 | 0.422 < u ≤ 0.475 | 2 |
| 0.6 | 0.581 < u ≤ 0.792 | 5 |
| 0.72 | 0.581 < u ≤ 0.792 | 5 |
| 0.81 | 0.792 < u ≤ 0.845 | 6 |
| 0.93 | 0.898 < u ≤ 0.951 | 8 |
| 0.99 | 0.951 < u ≤ 1.004 | 9 |

**Wynik resamplingu (krok 6):**

| | |
|--|--|
| wybrane i | `[0, 0, 0, 1, 2, 5, 5, 6, 8, 9]` |
| indeksy | `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]` |
| pozycja na świecie | `[4, 4, 1, 1, 3, 4, 1, 1, 1, 1]` |
| nowe cząsteczki | `[4, 4, 4, 4, 1, 4, 4, 1, 1, 1]` |


---
## Krok 7 — Trzeci ruch: +1 w prawo

`cząsteczki = (cząsteczki + 1) mod 5`

In [36]:
czasteczki = np.array([4, 4, 4, 4, 1, 4, 4, 1, 1, 1])  # po kroku 6

print("przed ruchem:", czasteczki.tolist())
czasteczki = move(czasteczki, len(swiat))
print("po ruchu +1: ", czasteczki.tolist())
print("rozkład:     ", rozklad(czasteczki, len(swiat)))

przed ruchem: [4, 4, 4, 4, 1, 4, 4, 1, 1, 1]
po ruchu +1:  [0, 0, 0, 0, 2, 0, 0, 2, 2, 2]
rozkład:      {0: 6, 2: 4}


---
## Krok 8 — Czwarty pomiar: ściana

Ściana na pozycjach `{0, 1, 3}`.

Wagi:
- `0.8` jeśli pole to ściana
- `0.2` w przeciwnym razie

In [37]:
czasteczki = np.array([0, 0, 0, 0, 2, 0, 0, 2, 2, 2])  # po kroku 7

pomiar = sciana

w = []
for pozycja in czasteczki:
    if swiat[pozycja] == pomiar:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)

S = w.sum()
w_prim = np.round(w / S, 3)

print("8.a")
print("S =", round(S, 2))
print("\n")

print("8.b")
print("w  (przed normalizacją) =", np.round(w, 3).tolist())

print("w' (po normalizacji)    =", w_prim.tolist())
print(f"suma w'= {round(w_prim.sum(), 3)} ≈ 1")

8.a
S = 5.6


8.b
w  (przed normalizacją) = [0.8, 0.8, 0.8, 0.8, 0.2, 0.8, 0.8, 0.2, 0.2, 0.2]
w' (po normalizacji)    = [0.143, 0.143, 0.143, 0.143, 0.036, 0.143, 0.143, 0.036, 0.036, 0.036]
suma w'= 1.002 ≈ 1


### Resampling (ruletka, u i CDF)

Losowe wartości:
`u = [0.05, 0.12, 0.20, 0.33, 0.45, 0.60, 0.72, 0.81, 0.93, 0.99]`

Dla każdego `u_j` wybieramy **najmniejszy indeks i**, dla którego `CDF_i ≥ u_j`.

In [38]:
czasteczki = np.array([0, 0, 0, 0, 2, 0, 0, 2, 2, 2])
w = []
for pozycja in czasteczki:
    if swiat[pozycja] == sciana:
        w.append(p_poprawny)
    else:
        w.append(p_bledny)
w = np.array(w)
w_prim = np.round(w / w.sum(), 3)

u = np.array([0.05, 0.12, 0.20, 0.33, 0.45, 0.60, 0.72, 0.81, 0.93, 0.99])
czasteczki, indeksy, CDF = resampling_ruletka(czasteczki, w_prim, u)

print("8.c Resampling (ruletka/CDF)")
print("w'  =", w_prim.tolist())
print("CDF ≈", np.round(CDF, 3).tolist())
print("wybrane indeksy i =", indeksy.tolist())
print("nowe cząsteczki   =", czasteczki.tolist())

from IPython.display import Markdown, display

linie = [
    "**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)",
    "",
    "| $u_j$ | Przedział CDF | Wybrany indeks $i$ |",
    "|------:|:--------------|-------------------:|",
]
for uj, i in zip(u, indeksy):
    prawy = float(CDF[i])
    if i == 0:
        przedzial = f"u ≤ {prawy:.3f}"
    else:
        lewy = float(CDF[i - 1])
        przedzial = f"{lewy:.3f} < u ≤ {prawy:.3f}"
    linie.append(f"| {uj} | {przedzial} | {int(i)} |")

display(Markdown("\n".join(linie)))

8.c Resampling (ruletka/CDF)
w'  = [0.143, 0.143, 0.143, 0.143, 0.036, 0.143, 0.143, 0.036, 0.036, 0.036]
CDF ≈ [0.143, 0.286, 0.429, 0.572, 0.608, 0.751, 0.894, 0.93, 0.966, 1.002]
wybrane indeksy i = [0, 0, 1, 2, 3, 4, 5, 6, 7, 9]
nowe cząsteczki   = [0, 0, 0, 0, 0, 2, 0, 0, 2, 2]


**Tabela:** Przyporządkowanie wartości $u_j$ do indeksów CDF (metoda ruletkowa)

| $u_j$ | Przedział CDF | Wybrany indeks $i$ |
|------:|:--------------|-------------------:|
| 0.05 | u ≤ 0.143 | 0 |
| 0.12 | u ≤ 0.143 | 0 |
| 0.2 | 0.143 < u ≤ 0.286 | 1 |
| 0.33 | 0.286 < u ≤ 0.429 | 2 |
| 0.45 | 0.429 < u ≤ 0.572 | 3 |
| 0.6 | 0.572 < u ≤ 0.608 | 4 |
| 0.72 | 0.608 < u ≤ 0.751 | 5 |
| 0.81 | 0.751 < u ≤ 0.894 | 6 |
| 0.93 | 0.894 < u ≤ 0.930 | 7 |
| 0.99 | 0.966 < u ≤ 1.002 | 9 |

**Wynik resamplingu (krok 8):**

| | |
|--|--|
| wybrane i | `[0, 0, 1, 2, 3, 4, 5, 6, 7, 9]` |
| indeksy | `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]` |
| pozycja na świecie | `[0, 0, 0, 0, 2, 0, 0, 2, 2, 2]` |
| nowe cząsteczki | `[0, 0, 0, 0, 0, 2, 0, 0, 2, 2]` |

### Wniosek


Po kolejnym ruchu `+1` i pomiarze **ściana** (krok 8) dominuje pole **0** (ściana) — `7/10` cząsteczek.

To się zgadza z trasą:

$$0 \xleftarrow{-1} 4 \xleftarrow{-1} 3 \xleftarrow{-1} 2$$

Start nadal **drzwi pod indeksem 2**. Pełne pomiary: drzwi → ściana → drzwi → ściana.
